# Data Space Interventions

## Group A: what evidence the learner receives

This notebook investigates whether the baseline fails because the available data or split design gives weak evidence for the project claim. Model family, optimizer, and input variables remain fixed.

By the end of this notebook you should be able to:

- distinguish a data-space intervention from a model or optimizer intervention;
- use validation evidence to argue for or against a data failure hypothesis;
- choose a data intervention without using the test set as a search tool;
- record a concise experiment result that your group can report back.

**Activity contract**

| Category | Rule |
|---|---|
| Fixed | Model family, inputs, optimizer, loss, evaluation report |
| Allowed | Split strategy, added-data policy, collection mix |
| Not allowed | Changing the model family, adding hidden variables, changing optimizer or loss |
| Selection rule | Choose interventions using validation evidence |
| Test rule | Use test performance only after the choice is fixed |

Keep `show_advanced=False` unless an extension is explicitly enabled.

Each main subsection follows this loop:

$$
H_{\mathrm{fail}}
\xrightarrow{E}
I_E
\xrightarrow{\Delta}
H_{\Delta}
\xrightarrow{\mathcal{E}}
R
\xrightarrow{J}
C.
$$

Here $H_{\mathrm{fail}}$ is the failure hypothesis, $E$ is evidence, $I_E$ is the interpretation of that evidence, $\Delta$ is the intervention design, $H_{\Delta}$ is the intervention hypothesis, $\mathcal{E}$ is the experiment, $R$ is the result, $J$ is the evaluation judgement, and $C$ is the conclusion or discussion claim.

Use the result log in each subsection to capture the evidence your group would report back.


In [ ]:
# Environment setup. The notebook is designed to run locally and in Colab.
import importlib.util
import os
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "nextgen2026-matplotlib")
)

REPO_URL = "https://github.com/nextgenerationgraduatesprogram/nextgen2026-mlai-workshops.git"
REPO_BRANCH = "workshop3"
PACKAGE_NAME = "nextgen2026_mlai_workshops"

if "google.colab" in sys.modules:
    repo_dir = Path("/content/nextgen2026-mlai-workshops")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPO_BRANCH,
                REPO_URL,
                str(repo_dir),
            ],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], check=True)
    missing_packages = [
        package_name
        for package_name, module_name in (("pandas", "pandas"), ("torch", "torch"))
        if importlib.util.find_spec(module_name) is None
    ]
    if missing_packages:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
    sys.path.insert(0, str(repo_dir / "src"))
else:
    repo_dir = None
    for possible_root in (Path.cwd(), Path.cwd().parent):
        possible_src = possible_root / "src"
        if (possible_src / PACKAGE_NAME).exists():
            repo_dir = possible_root
            sys.path.insert(0, str(possible_src))
            break

for module_name in list(sys.modules):
    if module_name == PACKAGE_NAME or module_name.startswith(f"{PACKAGE_NAME}."):
        del sys.modules[module_name]

print(f"Workshop 3 environment ready. Repository: {repo_dir or 'installed environment'}")


In [ ]:
# Load helpers and define small diagnostics for data-space comparisons.
from nextgen2026_mlai_workshops import solar_pv as pv
import matplotlib.pyplot as plt
import numpy as np

show_advanced = False
baseline_config = pv.baseline_config()
INPUT_COLUMNS = ["irradiance", "ambient_temperature", "tilt_angle"]

def require_filled_options(options, name):
    if any(option is Ellipsis for option in options):
        raise ValueError(
            f"Replace the ... placeholder in {name} with option lines before running this evidence cell."
        )


def input_split_summary(bundle, splits=("train", "validation")):
    rows = []
    for split in splits:
        data = bundle[split]
        rows.append(
            [
                split,
                len(pv.target_vector(data)),
                int(np.sum(pv.key_range_mask(data, bundle["criterion"]))),
                float(np.mean(data["irradiance"])),
                float(np.mean(data["ambient_temperature"])),
                float(np.mean(data["tilt_angle"])),
            ]
        )
    return rows


def plot_input_histograms(bundle, splits=("train", "validation"), title="Input distributions"):
    columns = INPUT_COLUMNS
    fig, axes = plt.subplots(1, len(columns), figsize=(12, 3.1))
    axes = np.atleast_1d(axes)
    for ax, column in zip(axes, columns):
        for split in splits:
            ax.hist(bundle[split][column], bins=18, alpha=0.45, label=split)
        ax.set_title(column.replace("_", " "))
        ax.set_xlabel(column.replace("_", " "))
        ax.set_ylabel("count")
        ax.grid(alpha=0.2)
    axes[0].legend()
    fig.suptitle(title, y=1.05)
    fig.tight_layout()
    return fig, axes


def plot_residual_vs_support(bundle, run):
    distances = pv.nearest_training_distance(bundle)
    validation = bundle["validation"]
    residuals = pv.predict_run(run, validation) - pv.target_vector(validation)
    fig, ax = plt.subplots(figsize=(5.8, 3.6))
    ax.scatter(distances, residuals, s=28, alpha=0.75, color="#7B5E9E", edgecolor="white", linewidth=0.3)
    ax.axhline(0.0, color="#333333", linewidth=1, linestyle="--")
    ax.set_xlabel("nearest training distance in input space")
    ax.set_ylabel("prediction residual")
    ax.set_title("Validation residuals by local support")
    ax.grid(alpha=0.2)
    fig.tight_layout()
    return fig, ax


pv.print_table(
    ["Category", "Setting"],
    [
        ["Fixed", "MLP family, inputs, plain SGD optimizer, MSE loss"],
        ["Allowed", "split evidence, added-data policy, collection mix"],
        ["Inputs", ", ".join(INPUT_COLUMNS)],
        ["Selection", "validation evidence first; test only after a choice is fixed"],
        ["Advanced diagnostics", show_advanced],
    ],
)


<br>

## 1. Split Design as an Evidence Protocol

### Motivation

Validation is the evidence we use to decide whether the model supports the project claim. If the validation split mostly resembles the training data, the limiting factor may not be the fitted model yet; it may be that the evidence protocol is asking an easier question than deployment will ask.

This section does **not** try to improve the trained model. It decides what question later interventions should be judged against.

### Failure Hypothesis

A random validation split may be too similar to the training distribution. If deployment depends on hotter, higher-output, or less common input regimes, random validation can under-sample those regimes and give weak evidence for the real claim.

### Evidence

We will compare several split designs using the same fixed modelling recipe. The evidence is validation-only: overall validation RMSE, key-range validation RMSE, and how many key-range examples each split places in train and validation. Because the model recipe is fixed, differences between rows mostly tell us how the validation question changed, not whether a data-collection intervention has repaired the model.


In [ ]:
# Compare split strategies while keeping the model recipe fixed.
split_results = pv.compare_split_strategies(seed=7, show_advanced=show_advanced)
pv.print_table(
    ["Split strategy", "Train key n", "Validation key n", "Validation RMSE", "Key range RMSE"],
    split_results["rows"],
)

print("\nInput-regime summaries by split strategy")
for strategy, summary_rows in split_results["summaries"].items():
    print(f"\n{strategy}")
    pv.print_table(
        ["Split", "n", "Key range n", "Mean irradiance", "Mean ambient", "Mean tilt"],
        summary_rows,
    )

split_strategy_options = tuple(row[0] for row in split_results["rows"])
print("\nSplit strategy options:", ", ".join(split_strategy_options))
print("No test metrics are shown in this evidence table.")


### Interpretation and Rationale

Changing the split does not repair the model. It changes the question the validation set asks. If a random split looks good while a deployment-structured split exposes larger errors, the evidence suggests that the original validation protocol may be too weak for decision-making.

Use the table to separate two ideas: which split gives the lowest error, and which split gives evidence that matches the deployment claim. A higher validation RMSE under a stricter split can be useful because it reveals where the current model claim is not yet supported.

Ask:

- Which split changes the number of deployment-relevant or key-range examples the most?
- Does the split with the easiest-looking metric actually test the claim you need to make?
- Are the weak slices caused by a worse model, or by a validation set that is finally looking at harder cases?
- What later intervention would each split make fair to evaluate?

### Intervention Hypothesis

A split protocol that deliberately tests deployment-relevant input regimes will be a stronger evidence protocol for later data-space decisions than an IID-looking random split. The expected improvement is not lower error in this subsection; it is clearer evidence about whether the current model can support the deployment claim.

### Experiment

Choose the split strategy your group would use as the evidence protocol. Set `selected_split_strategy` to one of the printed options only after your group can justify why that split tests the deployment claim. Do not treat this choice as a model fix; treat it as the validation question for later fixes.


In [ ]:
# Record the split strategy selected from validation evidence.
selected_split_strategy = None

if selected_split_strategy is None:
    print("Leave selected_split_strategy as None until your group has chosen an evidence protocol.")
    print("This locks the validation question; it does not add training data or improve the model.")
    print("Available options:", ", ".join(split_strategy_options))
elif selected_split_strategy not in split_strategy_options:
    raise ValueError(f"Choose one of: {', '.join(split_strategy_options)}")
else:
    selected_split_report = split_results["reports"][selected_split_strategy]
    print(f"Locked evidence protocol: {selected_split_strategy}")
    print("Use this as the validation question for later intervention choices.")
    pv.print_report(selected_split_report)


### Evaluation

Use the validation table to decide whether the split exposes or hides deployment-relevant weakness. There is no final test check in this subsection because the split choice is an evidence design, not a model or data-collection intervention.

### Result Log

| Experiment | Setup | Result | Conclusion |
|---|---|---|---|
| | | | |

### Conclusion / Discussion

What claim does your selected split let you test? What claim would still remain unsupported until you change the training data, model family, or optimizer?

Drill deeper:

- If two splits disagree, which one is more aligned with the intended deployment use?
- Does your chosen split make later intervention comparisons fairer or just harder?
- What additional slice or regime count would make the split choice more defensible?


<br>

## 2. Missing Combinations

### Motivation

In more than one input dimension, coverage along each separate axis can be misleading. The limiting factor may be local support: the model may be asked to predict for a combination of irradiance, temperature, and tilt that is far from anything it saw during training, even if each individual input value appears common.

### Failure Hypothesis

The training data may cover each input separately while still missing important combinations of inputs. A model can then make large errors where validation examples are far from the local training support, because the prediction is closer to extrapolation than interpolation.

### Evidence

Inspect input coverage, the shared validation report, nearest-training-example distance, and validation residuals. All diagnostics in this section use only the input columns and the target. The key question is whether large errors concentrate where nearby training evidence is weak.


In [ ]:
# Build the missing-combinations scenario and inspect support evidence.
missing_bundle = pv.make_workshop3_bundle("data_missing_combinations", seed=7)
missing_baseline = pv.train_with_config(missing_bundle, baseline_config, name="Missing-combinations baseline")

plot_input_histograms(
    missing_bundle,
    splits=("train", "validation"),
    title="Missing-combinations scenario: input coverage",
)

missing_report = pv.evaluate_model_report(
    missing_baseline,
    missing_bundle,
    include_test=False,
    show_advanced=show_advanced,
)
pv.print_report(missing_report)

print("\nWorst-supported validation examples")
pv.print_table(
    ["row", "nearest distance", "irradiance", "ambient temp", "tilt", "residual"],
    pv.low_support_rows(missing_bundle, missing_baseline, top_k=8),
)
plot_residual_vs_support(missing_bundle, missing_baseline)


### Interpretation and Rationale

Marginal coverage can look plausible while local support is weak in a joint region. Treat nearest-training distance as a proxy for support: it does not prove the cause, but it tells you where the model is relying on less local evidence than the one-dimensional histograms suggest.

If high residuals line up with high nearest-training distance, the evidence supports a data-support explanation. If high residuals occur in well-supported regions, the limiting factor may instead be the function class, objective, or an unobserved condition.

Ask:

- Are the largest residuals also among the lowest-support validation examples?
- Are low-support examples concentrated in a recognizable irradiance-temperature-tilt regime?
- Would adding more ordinary examples reduce the relevant distance, or would it leave the weak region mostly unchanged?
- Does a proposed collection policy reduce low-support RMSE, not just the average validation RMSE?

### Intervention Hypothesis

If the failure is an input-support problem, adding observations that improve support in weak or high-error input regions should improve validation evidence more directly than adding more common-condition examples. The intervention should be judged by whether the affected region improves, not only by whether the average score moves.

### Experiment

The next cell starts with `uniform_sampling` as a neutral control and leaves the targeted policies for your group to add. It also supports a custom option where you choose visible input values and collect extra rows at those settings. Keep the batch size fixed for the first pass so your comparison is about collection policy, not just more data.


In [ ]:
# Compare data-collection policies under the same evaluation protocol.
print("Built-in data_collection_options:", ", ".join(pv.DATA_COLLECTION_OPTIONS))
print("Custom option shape: {'label': 'my custom samples', 'input_values': custom_input_values}")

custom_input_values = [
    {"irradiance": 860.0, "ambient_temperature": 37.0, "tilt_angle": 8.0},
    {"irradiance": 860.0, "ambient_temperature": 37.0, "tilt_angle": 52.0},
]

data_collection_options = (
    "uniform_sampling",
    ...  # TODO: Add policies to test, e.g. "target_low_support" or "target_high_error"
    # TODO: Or add your own rows, e.g. {"label": "my custom samples", "input_values": custom_input_values}
)
require_filled_options(data_collection_options, "data_collection_options")

collection_results = pv.run_data_collection_options(
    missing_bundle,
    options=data_collection_options,
    added_n=200,
    seed=7,
    show_advanced=show_advanced,
)
pv.print_table(
    ["Policy", "Added n", "Validation RMSE", "Key range RMSE", "Low-support RMSE"],
    collection_results["rows"],
)

selected_collection_policy = None

if selected_collection_policy is None:
    print("Set selected_collection_policy after reviewing validation evidence.")
    print("Available options:", ", ".join(collection_results["runs"].keys()))
elif selected_collection_policy not in collection_results["runs"]:
    raise ValueError(f"Choose one of: {', '.join(collection_results['runs'])}")
else:
    selected_collection_run = collection_results["runs"][selected_collection_policy]
    selected_collection_bundle = collection_results["bundles"][selected_collection_policy]
    print(f"\nValidation report for fixed policy: {selected_collection_policy}")
    pv.print_report(
        pv.evaluate_model_report(
            selected_collection_run,
            selected_collection_bundle,
            include_test=False,
            show_advanced=show_advanced,
        )
    )
    selected_collection_report = pv.final_check(
        selected_collection_run,
        selected_collection_bundle,
        show_advanced=show_advanced,
    )
    print(f"\nFinal test check for fixed policy: {selected_collection_policy}")
    pv.print_report(selected_collection_report)


### Evaluation

Choose from the validation table, with particular attention to the low-support RMSE and the input slices in the shared report. Only reveal test performance after `selected_collection_policy` is fixed.

### Result Log

| Experiment | Setup | Result | Conclusion |
|---|---|---|---|
| | | | |

### Conclusion / Discussion

Did the evidence support a data-support explanation? Would duplicating existing rows fix the same problem, or would it leave the input gap mostly unchanged?

Drill deeper:

- Did the selected policy improve the weak region you diagnosed, or mainly improve easier examples?
- If a targeted policy does not help, does that weaken the support hypothesis or suggest the target region was chosen poorly?
- What evidence would distinguish missing support from missing input information?


<br>

## 3. Train/Deployment Mismatch

### Motivation

A training set teaches the model most strongly about the examples it contains most often. If deployment puts more weight on hotter, higher-output, or less common regimes, the limiting factor may be the sampling mix: the model has mostly been optimized for ordinary conditions while the project claim depends on different conditions.

Unlike section 1, the validation question is now treated as fixed deployment-facing evidence. This section asks whether changing the training evidence can improve performance on that fixed question.

### Failure Hypothesis

The training data may be weighted toward ordinary input regimes, while deployment needs reliable performance in hotter or less common irradiance-temperature-tilt regimes. A good average score can therefore hide a mismatch between the evidence used for training and the evidence needed for deployment.

### Evidence

Compare the train and validation input mixes, then inspect the baseline validation report. The validation split acts as the fixed deployment-facing evidence set here; test performance remains hidden until a collection mix is fixed. Look for regimes that are underrepresented in training and weak in validation.


In [ ]:
# Build the deployment-mismatch scenario and inspect train-validation differences.
mismatch_bundle = pv.make_workshop3_bundle("data_deployment_mismatch", seed=7)
mismatch_baseline = pv.train_with_config(mismatch_bundle, baseline_config, name="Deployment-mismatch baseline")

plot_input_histograms(
    mismatch_bundle,
    splits=("train", "validation"),
    title="Train versus deployment-validation input mix",
)

print("Input-regime summary")
pv.print_table(
    ["Split", "n", "Key range n", "Mean irradiance", "Mean ambient", "Mean tilt"],
    input_split_summary(mismatch_bundle, splits=("train", "validation")),
)

print("\nBaseline validation report")
pv.print_report(
    pv.evaluate_model_report(
        mismatch_baseline,
        mismatch_bundle,
        include_test=False,
        show_advanced=show_advanced,
    )
)


### Interpretation and Rationale

A model can perform acceptably on common training-like conditions while still being weak in the input regimes that matter for deployment. The suspected limiting factor is not simply dataset size; it is whether the added training evidence is weighted toward the regimes that decide the project claim.

Read the distribution summaries together with the slice metrics. If the weakest validation slice is also underrepresented in training, a deployment-matched or stratified collection mix is a direct test of the mismatch explanation. If the slice remains weak after rebalancing, the failure may need a hypothesis-space or objective-space explanation.

Ask:

- Which input regimes are overrepresented in training relative to the fixed deployment-facing validation set?
- Which deployment-relevant validation slice contributes most to the criterion failure?
- Does the proposed training-data mix improve that slice, or does it trade one weak regime for another?
- Would the same collection choice still look sensible if overall validation RMSE moved only slightly?

### Intervention Hypothesis

If deployment mismatch is the main data-space problem, changing the added-data mix toward deployment-relevant or visibly balanced regimes should improve validation evidence for the deployment claim. The rationale is that the model selection process sees more training examples from the conditions it must handle well.

### Experiment

The next cell starts with `normal_condition_mix` as a control and leaves deployment-facing mixes for your group to add. Keep the validation rows, model, and training recipe unchanged so the comparison isolates the data-collection choice.


In [ ]:
# Compare collection mixes for the deployment-mismatch scenario.
print("Built-in collection_mix_options:", ", ".join(pv.COLLECTION_MIX_OPTIONS))

collection_mix_options = (
    "normal_condition_mix",
    ...  # TODO: Add mixes to test, e.g. "deployment_matched_mix" or "input_regime_stratified_mix"
)
require_filled_options(collection_mix_options, "collection_mix_options")

mix_results = pv.run_collection_mix_options(
    mismatch_bundle,
    options=collection_mix_options,
    added_n=300,
    seed=7,
    show_advanced=show_advanced,
)
pv.print_table(
    ["Collection mix", "Added n", "Validation RMSE", "Key range RMSE", "Criterion pass"],
    mix_results["rows"],
)

selected_collection_mix = None

if selected_collection_mix is None:
    print("Set selected_collection_mix after reviewing validation evidence.")
    print("Available options:", ", ".join(mix_results["runs"].keys()))
elif selected_collection_mix not in mix_results["runs"]:
    raise ValueError(f"Choose one of: {', '.join(mix_results['runs'])}")
else:
    selected_mix_run = mix_results["runs"][selected_collection_mix]
    selected_mix_bundle = mix_results["bundles"][selected_collection_mix]
    print(f"\nValidation report for fixed collection mix: {selected_collection_mix}")
    pv.print_report(
        pv.evaluate_model_report(
            selected_mix_run,
            selected_mix_bundle,
            include_test=False,
            show_advanced=show_advanced,
        )
    )
    selected_mix_report = pv.final_check(
        selected_mix_run,
        selected_mix_bundle,
        show_advanced=show_advanced,
    )
    print(f"\nFinal test check for fixed collection mix: {selected_collection_mix}")
    pv.print_report(selected_mix_report)


### Evaluation

Choose the collection mix using validation evidence, not test performance. Compare overall validation RMSE with the key-range and slice evidence so your conclusion does not depend on a single average score. This is different from section 1: here the validation question stays fixed and the training data changes.

### Result Log

| Experiment | Setup | Result | Conclusion |
|---|---|---|---|
| | | | |

### Conclusion / Discussion

Which collection mix gives evidence that better supports the deployment claim? What tradeoff would you tell another group to check before treating the intervention as generally reliable?

Drill deeper:

- Did the mix improve the deployment-relevant slice or only the overall average?
- Which common-condition performance, if any, was traded away?
- If the mismatch explanation is only partly supported, which next diagnostic would you request?
